# 🎬 VP Library — Embed ภาพลง Supabase

Notebook นี้ embed ภาพทุกฉากด้วย **mCLIP** ผ่าน HuggingFace API แล้วบันทึก vector ลง Supabase

### วิธีใช้
1. ใส่ `SUPABASE_SERVICE_KEY` และ `HF_TOKEN` ใน Cell 2
2. กด **Runtime → Run all**
3. รอ ~3–5 นาที แล้วเสร็จ

> **HF Token** — สร้างได้ที่ https://huggingface.co/settings/tokens (ฟรี, ใช้ Read permission)

In [ ]:
# ── Cell 1: ติดตั้ง package ──────────────────────────────────────
!pip install supabase requests -q
print('✅ พร้อม')

In [ ]:
# ── Cell 2: ตั้งค่า — ใส่ key ทั้งสองตรงนี้ ─────────────────────

SUPABASE_URL         = 'https://pgaqdqbjyewwckpslyvx.supabase.co'
SUPABASE_SERVICE_KEY = 'YOUR_SERVICE_ROLE_KEY_HERE'  # Supabase → Settings → API → Secret keys
HF_TOKEN             = 'YOUR_HF_TOKEN_HERE'          # https://huggingface.co/settings/tokens

SKIP_ALREADY_INDEXED = True   # True = ข้าม scene ที่ embed แล้ว (แนะนำ)

# ── ตรวจสอบ ──
ok = True
if 'YOUR_SERVICE' in SUPABASE_SERVICE_KEY:
    print('⚠️  ยังไม่ได้ใส่ SUPABASE_SERVICE_KEY'); ok = False
if 'YOUR_HF' in HF_TOKEN:
    print('⚠️  ยังไม่ได้ใส่ HF_TOKEN'); ok = False
if ok:
    print('✅ Config พร้อม')

In [ ]:
# ── Cell 3: เชื่อมต่อ Supabase + ทดสอบ HF API ───────────────────
import requests, time
from supabase import create_client
from datetime import datetime, timezone

# เชื่อมต่อ Supabase
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print('✅ Supabase เชื่อมต่อแล้ว')

# หา HF endpoint ที่ใช้งานได้
HF_URLS = [
    'https://router.huggingface.co/hf-inference/models/sentence-transformers/clip-ViT-B-32-multilingual-v1',
    'https://router.huggingface.co/pipeline/feature-extraction/sentence-transformers/clip-ViT-B-32-multilingual-v1',
    'https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/clip-ViT-B-32-multilingual-v1',
]
HF_HEADERS = {
    'Authorization': f'Bearer {HF_TOKEN}',
    'Content-Type': 'application/json',
}

HF_WORKING_URL = None
print('🔄 ทดสอบ HF API endpoint...')
for url in HF_URLS:
    try:
        r = requests.post(url, headers=HF_HEADERS,
                          json={'inputs': 'test', 'options': {'wait_for_model': True}},
                          timeout=30)
        if r.status_code == 200:
            data = r.json()
            vec  = data[0] if isinstance(data[0], list) else data
            if isinstance(vec, list) and len(vec) > 100:
                HF_WORKING_URL = url
                print(f'✅ HF URL ใช้งานได้: {url}')
                print(f'   vector dimension: {len(vec)}')
                break
        elif r.status_code == 503:
            print(f'   503 model cold start, รอ 10s...')
            time.sleep(10)
            r = requests.post(url, headers=HF_HEADERS,
                              json={'inputs': 'test', 'options': {'wait_for_model': True}},
                              timeout=30)
            if r.status_code == 200:
                data = r.json()
                vec  = data[0] if isinstance(data[0], list) else data
                HF_WORKING_URL = url
                print(f'✅ HF URL ใช้งานได้ (หลัง warm up): {url}')
                break
        else:
            print(f'   ❌ {url} → {r.status_code}')
    except Exception as e:
        print(f'   ❌ {url} → {e}')

if not HF_WORKING_URL:
    print('\n❌ ไม่พบ HF URL ที่ใช้งานได้ — ตรวจสอบ HF_TOKEN')
else:
    print(f'\n🎯 จะใช้: {HF_WORKING_URL}')

In [ ]:
# ── Cell 4: ดึงรายการฉากจาก Supabase ──────────────────────────────
assert HF_WORKING_URL, 'HF URL ไม่พร้อม — รัน Cell 3 ใหม่'

result    = sb.from_('scenes').select('id, image_url, mclip_indexed_at').order('sort_order').execute()
all_sc    = result.data or []
to_embed  = [
    s for s in all_sc
    if s.get('image_url') and (not SKIP_ALREADY_INDEXED or not s.get('mclip_indexed_at'))
]

print(f'📊 ฉากทั้งหมด:    {len(all_sc)}')
print(f'✅ embed แล้ว:   {len(all_sc) - len(to_embed)}')
print(f'⏳ รอ embed:     {len(to_embed)}')
if not to_embed:
    print('\n🎉 ทุกฉาก embed แล้ว!')

In [ ]:
# ── Cell 5: Embed ทุกฉาก → บันทึกลง Supabase ───────────────────────
def encode_image(image_url: str, retries=3) -> list:
    """ดาวน์โหลดภาพ → ส่ง raw bytes ไป HF API → คืน vector[512]"""
    img_bytes = requests.get(image_url, timeout=30).content
    for attempt in range(retries):
        # ส่ง raw bytes (Python ไม่มีปัญหา CORS ต่างจาก browser)
        r = requests.post(
            HF_WORKING_URL,
            headers={'Authorization': f'Bearer {HF_TOKEN}'},
            data=img_bytes,
            timeout=60,
        )
        if r.status_code == 503:
            print(f'     model cold start, รอ 15s...')
            time.sleep(15); continue
        r.raise_for_status()
        data = r.json()
        vec  = data[0] if isinstance(data[0], list) else data
        assert isinstance(vec, list) and len(vec) > 100, f'vector ผิดรูปแบบ: {type(vec)}'
        return vec
    raise Exception('ล้มเหลวหลัง retry')

if not to_embed:
    print('🎉 ไม่มีฉากที่ต้อง embed')
else:
    success, failed, errors = 0, 0, []
    for i, scene in enumerate(to_embed):
        sid = scene['id']
        print(f'[{i+1}/{len(to_embed)}] 🔄 {sid}...', end=' ', flush=True)
        try:
            vector = encode_image(scene['image_url'])
            now    = datetime.now(timezone.utc).isoformat()
            sb.from_('scenes').update({
                'mclip_vector':     vector,
                'mclip_indexed_at': now,
            }).eq('id', sid).execute()
            success += 1
            print(f'✅ dim={len(vector)}')
        except Exception as e:
            failed += 1
            errors.append((sid, str(e)))
            print(f'❌ {e}')

    print(f'\n══════════════════════════')
    print(f'✅ สำเร็จ:  {success} ฉาก')
    print(f'❌ ล้มเหลว: {failed} ฉาก')
    if errors:
        for sid, err in errors: print(f'   {sid}: {err}')
    if success > 0:
        print('\n🎉 เสร็จ! ตอนนี้ vp-search.html ค้นหาได้แล้ว')

In [ ]:
# ── Cell 6 (Optional): ตรวจสอบผลลัพธ์ใน Supabase ──────────────────
rows      = (sb.from_('scenes').select('id, mclip_indexed_at').order('sort_order').execute()).data or []
indexed   = [r for r in rows if r.get('mclip_indexed_at')]
unindexed = [r for r in rows if not r.get('mclip_indexed_at')]
print(f'✅ มี vector: {len(indexed)} ฉาก')
print(f'⏳ ยังไม่มี:  {len(unindexed)} ฉาก')
if unindexed:
    for r in unindexed: print(f'   • {r["id"]}')
else:
    print('\n🎉 ทุกฉากครบ!')